# CSV FILES
We have four Files: 
* `HSall_members.csv`: The Member Ideology data export contains biographical and ideological information about members of congress for the chamber(s) and congress(es) that were selected.
* `HSall_parties.csv`: The Congressional Party data export contains biographical information and ideological mean and median scores for congressional parties in the selected chamber(s) and congress(es).
* `HSall_rollcalls.csv`: contains data about each rollcall taken by Congress in the selected chamber(s) and congress(es). Note that this data refers to the overall result and ideological parameters of the vote, not any individual member’s positions.
* `HSall_votes.csv`: This data contains basic information about how each member in the selected chamber(s) and congress(es) voted on each vote.

Their Datafields are described in detail in the next section.
For now it's just important that each file has collumn `congress` which gives us a easy way to approximately filter the data by year:
* `congress`: Integer 1+. The number of the congress that this file’s row refers to. e.g. 115 for the 115th Congress (2017-2019)`

In [1]:
%matplotlib ipympl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

#fig, ax = plt.subplots()

# data filtering
# we need 1990-2011 which is included in these congress periods:
#congress_numbers_1989_2013 = np.arange(102,112+1)
#def congress_filter(row):
#    return row['congress'].isin(congress_numbers_1989_2013)


##### LODAING THE DATA #####

# important note is you downloaded the data yourself:
#   in some lines of the votes.prob collumn there is "N/AN/AN/AN/AN/A..." written in a float field
#   either we build a parser for this collumn or you simply replace these wrong strings with just "NA"

# data types datatypes are needed otherwise the parser won't handle
# NA entries.
members_types = {
    'congress':'Int64',
    'chamber':str,
    'icpsr':'Int64',
    'state_icpsr':'Int64',
    'district_code':'Int64',
    'state_abbrev':str,
    'party_code':'Int64',
    'occupancy':'Int64',
    'last_means':'Int64',
    'bioname':str,
    'bioguide_id':str,
    'born':'Int64',
    'died':'Int64',
    'nominate_dim1':float,
    'nominate_dim2':float,
    'nominate_log_likelihood':float,
    'nominate_geo_mean_probability':float,
    'nominate_number_of_votes':'Int64',
    'nominate_number_of_errors':'Int64',
    'conditional':'Int64',
    'nokken_poole_dim1':float,
    'nokken_poole_dim2':float,
}
parties_types = {
    'congress':'Int64',
    'chamber':str,
    'party_code':'Int64',
    'party_name':str,
    'n_members':'Int64',
    'nominate_dim1_median':float,
    'nominate_dim2_median':float,
    'nominate_dim1_mean':float,
    'nominate_dim2_mean':float,
}
rollcalls_types = {
    'congress':'Int64',
    'chamber':str,
    'rollnumber':'Int64',
    'date':str,    # maybe change to date. Format is: YYYY-mm-dd
    'session':'Int64',
    'clerk_rollnumber':'Int64',
    'yea_count':'Int64',
    'nay_count':'Int64',
    'nominate_mid_1':float,
    'nominate_mid_2':float,
    'nominate_spread_1':float,
    'nominate_spread_2':float,
    'nominate_log_likelihood':float,
    'bill_number':str,
    'vote_result':str,
    'vote_desc':str,
    'vote_question':str,
    'dtl_desc':str
}
votes_types = {
    'congress':'Int64',
    'chamber':str,
    'rollnumber':'Int64',
    'icpsr':'Int64',
    'cast_code':'Int64',
    #'prob':float,
}

df_members   = (pd.read_csv("../data/HSall_members.csv",dtype=members_types))#[congress_filter])
df_parties   = (pd.read_csv("../data/HSall_parties.csv",dtype=parties_types))#[congress_filter])
df_rollcalls = (pd.read_csv("../data/HSall_rollcalls.csv",dtype=rollcalls_types))#[congress_filter])
df_votes     = (pd.read_csv("../data/HSall_votes.csv",dtype=votes_types))#[congress_filter])

print(f"members csv {df_members.shape[0]} rows:\n",df_members.keys())
print(f"parties csv {df_parties.shape[0]} rows:\n",df_parties.keys())
print(f"rollcalls csv {df_rollcalls.shape[0]} rows:\n",df_rollcalls.keys())
print(f"votes csv {df_votes.shape[0]} rows:\n",df_votes.keys())

members csv 51060 rows:
 Index(['congress', 'chamber', 'icpsr', 'state_icpsr', 'district_code',
       'state_abbrev', 'party_code', 'occupancy', 'last_means', 'bioname',
       'bioguide_id', 'born', 'died', 'nominate_dim1', 'nominate_dim2',
       'nominate_log_likelihood', 'nominate_geo_mean_probability',
       'nominate_number_of_votes', 'nominate_number_of_errors', 'conditional',
       'nokken_poole_dim1', 'nokken_poole_dim2'],
      dtype='str')
parties csv 848 rows:
 Index(['congress', 'chamber', 'party_code', 'party_name', 'n_members',
       'nominate_dim1_median', 'nominate_dim2_median', 'nominate_dim1_mean',
       'nominate_dim2_mean'],
      dtype='str')
rollcalls csv 113369 rows:
 Index(['congress', 'chamber', 'rollnumber', 'date', 'session',
       'clerk_rollnumber', 'yea_count', 'nay_count', 'nominate_mid_1',
       'nominate_mid_2', 'nominate_spread_1', 'nominate_spread_2',
       'nominate_log_likelihood', 'bill_number', 'vote_result', 'vote_desc',
       'vote_que

/var/folders/zs/b5h2h6097nb8d9lmbw5vx90w0000gn/T/ipykernel_52741/1802961245.py:90: DtypeWarning: Columns (0: prob) have mixed types. Specify dtype option on import or set low_memory=False.
  df_votes     = (pd.read_csv("../data/HSall_votes.csv",dtype=votes_types))#[congress_filter])


In [2]:
df_votes.drop(df_votes.columns[df_votes.columns.tolist().index('prob')], axis=1, inplace=True)


# The core PK/FK connections between files:
* `members.icpsr` 1 <-> N `votes.icpsr`:
* `members.party_code` N <-> M `parties.party_code`:
* `votes.congress, votes.chamber, votes.rollnumber` N <-> 1 `rollcalls.congress, rollcalls.chamber, rollcalls.rollnumber`



## member fields
### Biographical Fields:
* **congress**: Integer 1+. The number of the congress that this member’s row refers to. e.g. 115 for the 115th Congress (2017-2019)
* **chamber**: House, Senate, or President. The chamber in which the member served.
* **icpsr**: Integer 1-99999. This is an ID code which identifies the member in question. In general, each member receives a single ICPSR identifier applicable to their entire career. A small number of members have received more than one: this can occur for members who have switched parties; as well as members who subsequently become president. Creating a new identifier allows a new NOMINATE estimate to be produced for separate appearances of a member in different roles.
* **state_icpsr**: Integer 0-99. Identifier for the state represented by the member.
* **district_code**: Integer 0-99. Identifier for the district that the member represents within their state (e.g. 3 for the Alabama 3rd Congressional District). Senate members are given district_code 0. Members who represent historical “at-large” districts are assigned 99, 98, or 1 in various circumstances.
* **state_abbrev**: String. Two-character postal abbreviation for state (e.g. MO for Missouri).
* **party_code**: Integer 1-9999. Identifying code for the member’s party. Please see documentation for Party Data for more information about which party_code identifiers refer to which parties.
* **occupancy**: Integer 1+. ICPSR occupancy code. This item is considered legacy or incomplete information and has not been verified. In general, members receive 0 if they are the only occupant, 1 if they are the first occupant, 2 if they are the second occupant, etc.
* **last_means**: Integer 1-5. ICPSR Attain-Office Code. This is an indicator that reflects the member’s last means of attaining office. This item is considered legacy or incomplete information and has not been verified. Members received 1 if they were elected in a general election, 2 if elected by special election, 3 if directly elected by a state legislature, and 5 if appointed.
* **bioname**: String. Name of the member, surname first. For most members, agrees with the Biographical Directory of Congress.
* **bioguide_id**: String. Member identifier in the Biographical Directory of Congress.
* **born**: Integer. Year of member’s birth.
* **died**: Integer. Year of member’s death.
### Ideological Fields:
* **nominate_dim1**: NOMINATE first dimension estimate.
* **nominate_dim2**: NOMINATE second dimension estimate.
* **log_likelihood**: Log-likelihood of the NOMINATE estimate.
* **geo_mean_probability**: Geometric mean probability of NOMINATE estimate.
* **number_of_votes**: Number of votes cast by the member during a given congress.
* **conditional**: Integer 0-1. A 1 indicates NOMINATE was estimated conditionally for a given member. 0 otherwise. Conditional estimation implies that an estimate is provisional and subject to updates when the next full estimation of NOMINATE scores occurs.
* **nokken_poole_dim1**: Nokken-Poole First dimension estimate.
* **nokken_poole_dim2**: Nokken-Poole Second dimension estimate.

## Partie fields
### Biographical Fields:
* **congress**: Integer 1+. The number of the congress that this member’s row refers to. e.g. 115 for the 115th Congress (2017-2019)
* **chamber**: House, Senate, or President. The chamber in which the member served.
* **party_code**: Integer 1-9999. Identifying code for the member’s party. See below for a list mapping party codes to parties.
* **party_name**: String. The name of the party.
* **n_members**: Integer 1+. Number of party members in this chamber in this congress.
### Ideological Fields:
* **dim1_median**: Median of the NOMINATE first dimension estimates among members of party_code in this chamber during congress.
* **dim2_median**: Median of the NOMINATE second dimension estimates among members of party_code in this chamber during congress.
* **dim1_median**: Mean of the NOMINATE first dimension estimates among members of party_code in this chamber during congress.
* **dim2_median**: Mean of the NOMINATE second dimension estimates among members of party_code in this chamber during congress.


## rollcalls fields
### Informational Fields:
* **congress**: Integer 1+. The number of the congress that this member’s row refers to. e.g. 115 for the 115th Congress (2017-2019)
* **chamber**: House, Senate, or President. The chamber in which the member served.
* **rollnumber**: Integer 1+. Starts from 1 in the first rollcall of each congress. Excludes quorum calls and vacated votes. This number does not match the clerk_rollnumber field.
* **date**: Date on which the rollcall took place.
* **session**: Integer 1 or 2. Indicator for which session of the congress in which a rollcall occurred.
* **clerk_rollnumber**: Number assigned by Congress to the rollcall. Includes quorum calls and vacated votes, starts from 1 at the beginning of the session (not at the beginning of the congress).
* **bill_number**: String. Number of the bill on which rollcall was held.
* **vote_result**: Official result of rollcall, may not exist for all votes.
* **vote_desc**: Description of the rollcall assigned by Congressional staff.
* **vote_question**: String. Question addressed by the rollcall. e.g. “On Agreeing to the Amendment”.
* **dtl_desc**: Description of the rollcall, collected from historical sources.
### Ideological Fields:
* **mid_1**: NOMINATE First-dimension midpoint estimate.
* **mid_2**: NOMINATE Second-dimension midpoint estimate.
* **spread_1**: NOMINATE First-dimension spread estimate.
* **spread_2**: NOMINATE Second-dimension spread estimate.
* **log_likelihood**: NOMINATE estimated log-likelihood.


## votes fields:
### Fields:
* **congress**: Integer 1+. The number of the congress that this member’s row refers to. e.g. 115 for the 115th Congress (2017-2019)
* **chamber**: House, Senate, or President. The chamber in which the member served.
* **rollnumber**: Integer 1+. Starts from 1 in the first rollcall of each congress. Excludes quorum calls and vacated votes.
* **icpsr**: Integer 1-99999. This is an ID code which identifies the member in question. In general, each member receives a single ICPSR identifier applicable to their entire career. A small number of members have received more than one: this can occur for members who have switched parties; as well as members who subsequently become president. Creating a new identifier allows a new NOMINATE estimate to be produced for separate appearances of a member in different roles.
* **cast_code**: Integer 0-9. Indicator of how the member voted.
* **prob**: Estimated probability, based on NOMINATE, of the member making the vote as recorded.

### Cast Codes
|cast_code|	Description|
|---------|------------|
|0|	Not a member of the chamber when this vote was taken|
|1|	Yea|
|2|	Paired Yea|
|3|	Announced Yea|
|4|	Announced Nay|
|5|	Paired Nay|
|6|	Nay|
|7|	Present (some Congresses)|
|8|	Present (some Congresses)|
|9|	Not Voting (Abstention)|

## Minimal Dataset for Paper reproduction

The idea is to combine the important columns into one table and throw away the rest

In [ ]:
df_data = df_votes

df_data = df_data.merge(df_rollcalls, on=["congress", "chamber", "rollnumber"],
    how="inner")

df_data = df_data.merge(df_members,  on=["congress", "chamber", "icpsr"],
    how="inner")

df_data = df_data.merge(df_parties,  on=["congress", "chamber", "party_code"], how="inner")

df_data = df_data[df_data['chamber'] == 'House']

df_data = df_data[['icpsr', 'cast_code', 'date', 'party_name', 'bioname', 'congress', 'rollnumber']]

df_data.head()

,icpsr,cast_code,date,party_name,bioname
0,154,6,1789-05-16,Pro-Administration,"AMES, Fisher"
1,259,9,1789-05-16,Anti-Administration,"ASHE, John Baptista"
2,379,1,1789-05-16,Anti-Administration,"BALDWIN, Abraham"
3,649,1,1789-05-16,Pro-Administration,"BENSON, Egbert"
4,786,1,1789-05-16,Anti-Administration,"BLAND, Theodorick"


Next we will translate the cast code column into the same structure used in the paper, meaning 1 for Yay, -1 for Nay, and 0 for abstained. This means we map labels 1,2,3 to 1, labels 4,5,6, to -1, labels 7,8,9 to 0 and label 0 to NA since we have no corresponding number in the paper. 

In [6]:
def map_cast_code(x):
    if x in [1, 2, 3]:
        return 1
    if x in [4, 5, 6]:
        return -1
    if x in [7, 8, 9]:
        return 0
    return pd.NA

df_data["paper_cast_code"] = df_data["cast_code"].apply(map_cast_code)

In [7]:
df_data.describe()

,icpsr,cast_code,paper_cast_code
count,21746511.0,21746511.0,2.174651e+07
mean,14488.621618,3.634182,2.214724e-01
std,10018.194324,3.01824,9.070253e-01
min,1.0,1.0,-1.000000e+00
25%,7026.0,1.0,-1.000000e+00
50%,14239.0,1.0,1.000000e+00
75%,20924.0,6.0,1.000000e+00
max,99999.0,9.0,1.000000e+00
